# BlitzMart Analytics: Python Analysis

Built on the cleaned dataset and the SQL findings. 12 analytical techniques that supply chain and commercial teams actually use:

- **ABC analysis** (products + customers). The 80/20 rule applied to inventory and customer value.
- **Demand trend & seasonality.** Monthly patterns and the Q4 spike.
- **OTIF carrier scorecard.** On-Time-In-Full performance ranking.
- **Warehouse fulfilment speed.**
- **Return root cause.** Rates by category, top reasons.
- **Forecast vs Actual.** Demand planning accuracy.
- **Stockout risk proxy.** High-velocity SKUs.
- **Customer geography.** Revenue by German city.

Each section produces a summary table and a chart. The final stages export 11 CSVs for the Tableau dashboards and auto-generate a findings summary.

## 1. Setup

### 1.1 Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Imports + chart defaults

pandas and numpy for data work, matplotlib and seaborn for charts. The rcParams calls below set consistent figure size, font size, and grid style across every chart in the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# consistent style for every chart in the notebook
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")
sns.set_palette("Set2")

print("Libraries loaded")

## 2. Load Cleaned Data

Reading all 8 cleaned tables from `data_clean/`. `parse_dates` is set so date columns load as proper datetime types and not strings, which makes the date math later much easier.

In [ ]:
CLEAN = "/content/drive/MyDrive/BlitzMart_Analytics/data_clean"

orders      = pd.read_csv(f"{CLEAN}/orders.csv", parse_dates=["order_date"])
order_items = pd.read_csv(f"{CLEAN}/order_items.csv")
products    = pd.read_csv(f"{CLEAN}/products.csv")
customers   = pd.read_csv(f"{CLEAN}/customers.csv", parse_dates=["signup_date"])
warehouses  = pd.read_csv(f"{CLEAN}/warehouses.csv")
shipments   = pd.read_csv(f"{CLEAN}/shipments.csv", parse_dates=["ship_date", "delivery_date"])
returns     = pd.read_csv(f"{CLEAN}/returns.csv", parse_dates=["return_date"])
suppliers   = pd.read_csv(f"{CLEAN}/suppliers.csv")

for name, df in [("orders", orders), ("order_items", order_items),
                 ("products", products), ("customers", customers),
                 ("shipments", shipments), ("returns", returns)]:
    print(f"  {name:<15} {len(df):>10,} rows")

## 3. Refresh Customer Geography

The customers in the generated data were spread evenly across all 15 cities. That doesn't match reality. Berlin has 3.7M people, Duisburg has 500K, they shouldn't show the same customer count.

This cell redistributes customers by real population shares (approx Statistisches Bundesamt 2024). Customer IDs don't change, only `city` and `region` get reassigned. All orders, shipments, returns stay intact.

One-time fix. Once it runs, the city revenue analysis later in the notebook gives realistic results with Berlin dominating.

In [ ]:
np.random.seed(42)

# population shares from Statistisches Bundesamt (approx 2024)
# Berlin gets the largest share, biggest population + strongest online buying base
city_weights = {
    "Berlin":             0.22,
    "Hamburg":            0.13,
    "München":            0.12,
    "Köln":               0.08,
    "Frankfurt am Main":  0.07,
    "Stuttgart":          0.06,
    "Düsseldorf":         0.05,
    "Leipzig":            0.05,
    "Dortmund":           0.04,
    "Essen":              0.04,
    "Bremen":             0.04,
    "Dresden":            0.03,
    "Hannover":           0.03,
    "Nürnberg":           0.02,
    "Duisburg":           0.02,
}

city_regions = {
    "Berlin": "East", "Hamburg": "North", "München": "South", "Köln": "West",
    "Frankfurt am Main": "West", "Stuttgart": "South", "Düsseldorf": "West",
    "Leipzig": "East", "Dortmund": "West", "Essen": "West", "Bremen": "North",
    "Dresden": "East", "Hannover": "North", "Nürnberg": "South", "Duisburg": "West",
}

probs = np.array(list(city_weights.values()))
probs /= probs.sum()

new_cities = np.random.choice(list(city_weights.keys()), size=len(customers), p=probs)
customers["city"]   = new_cities
customers["region"] = customers["city"].map(city_regions)

# write back so other notebooks see the redistributed data
customers.to_csv(f"{CLEAN}/customers.csv", index=False)

distribution = customers["city"].value_counts(normalize=True).mul(100).round(1)
print("Customer cities redistributed by population weight:\n")
print(distribution.to_string())

## 4. Build the Master DataFrame

Most of the analyses below need one flat, denormalised view. Building it once and reusing it is cleaner than re-merging tables in every cell.

Two things worth flagging here:

- **`suffixes=("", "_product")` on the products merge.** Both `order_items` and `products` have a `unit_price` column but they mean different things. `order_items.unit_price` is what the customer paid at the time of sale. `products.unit_price` is the current catalogue price. Revenue analysis should use the transaction price, so I keep that as the unqualified column.
- **Status filter.** Cancelled and Processing orders are excluded (no realised revenue). Delivered, Shipped, Returned are all kept (a return is still a completed sale, just refunded later).

Two derived columns added: `gross_profit` and `order_month`.

In [ ]:
# only orders that moved through fulfilment count for revenue
valid_orders = orders[orders["order_status"].isin(["Delivered", "Shipped", "Returned"])]

df = order_items.merge(valid_orders, on="order_id", how="inner")
df = df.merge(products, on="product_id", how="left", suffixes=("", "_product"))
df = df.merge(warehouses, on="warehouse_id", how="left")

# derived columns used throughout
df["revenue"]      = df["line_total"]
df["gross_profit"] = (df["unit_price"] - df["unit_cost"]) * df["quantity"] * (1 - df["discount"])
df["order_month"]  = df["order_date"].dt.to_period("M")

print(f"Master DataFrame: {len(df):,} rows, {df.shape[1]} columns")
df.head(3)

## 5. ABC Analysis: Products

ABC is the standard inventory technique. Applies the Pareto (80/20) principle to product revenue:

- **A items.** First 80% of cumulative revenue. Need tight stock control, premium fulfilment, zero stockout tolerance.
- **B items.** Next 15% (from 80% to 95% cumulative). Standard management.
- **C items.** Final 5%. Lean inventory, fewer review cycles.

Sort products by revenue descending, compute each one's cumulative share, assign class based on which threshold the cumulative percentage crosses.

In [ ]:
# sort revenue desc so cumsum measures "everything down to here"
abc_products = df.groupby(["product_id", "product_name", "category"], as_index=False)["revenue"].sum()
abc_products = abc_products.sort_values("revenue", ascending=False).reset_index(drop=True)

abc_products["cum_revenue"]     = abc_products["revenue"].cumsum()
abc_products["cum_revenue_pct"] = abc_products["cum_revenue"] / abc_products["revenue"].sum()

# thresholds: 80% cuts A, 95% cuts B, rest is C
def abc_class(pct):
    if pct <= 0.80: return "A"
    elif pct <= 0.95: return "B"
    else: return "C"

abc_products["abc_class"] = abc_products["cum_revenue_pct"].apply(abc_class)

# summary per class
abc_summary = abc_products.groupby("abc_class").agg(
    product_count=("product_id", "count"),
    revenue_eur=("revenue", "sum"),
).reset_index()
abc_summary["pct_of_products"] = (100 * abc_summary["product_count"] / len(abc_products)).round(1)
abc_summary["pct_of_revenue"]  = (100 * abc_summary["revenue_eur"] / abc_summary["revenue_eur"].sum()).round(1)
abc_summary

Pareto curve. Cumulative revenue % against product rank. Horizontal lines at 80% and 95% mark the A/B and B/C boundaries.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(len(abc_products)), 100 * abc_products["cum_revenue_pct"],
        color="#2E5266", linewidth=2)
ax.axhline(80, color="green", linestyle="--", alpha=0.6, label="80% revenue line")
ax.axhline(95, color="orange", linestyle="--", alpha=0.6, label="95% revenue line")
ax.set_xlabel("Products ranked by revenue (highest first)")
ax.set_ylabel("Cumulative revenue %")
ax.set_title("Product ABC: Pareto curve")
ax.legend()
plt.tight_layout()
plt.show()

## 6. ABC Analysis: Customers

Same 80/20 logic but for customers. A customers are the revenue backbone (losing one of them hurts disproportionately). C customers transact but bring little individual value.

Reuses the `abc_class()` function from Section 5 since the logic is identical.

In [ ]:
customer_revenue = df.groupby("customer_id", as_index=False)["revenue"].sum()
customer_revenue = customer_revenue.sort_values("revenue", ascending=False).reset_index(drop=True)
customer_revenue["cum_pct"] = customer_revenue["revenue"].cumsum() / customer_revenue["revenue"].sum()
customer_revenue["abc_class"] = customer_revenue["cum_pct"].apply(abc_class)

cust_abc_summary = customer_revenue.groupby("abc_class").agg(
    customers=("customer_id", "count"),
    revenue_eur=("revenue", "sum"),
).reset_index()
cust_abc_summary["pct_of_customers"] = (100 * cust_abc_summary["customers"] / len(customer_revenue)).round(1)
cust_abc_summary["pct_of_revenue"]   = (100 * cust_abc_summary["revenue_eur"] / cust_abc_summary["revenue_eur"].sum()).round(1)
cust_abc_summary

## 7. Demand Trend & Seasonality

Aggregate units, revenue, and orders by month. The plot makes the Nov/Dec spike obvious. Identifying this pattern is a prerequisite for any forecasting or capacity planning.

In [ ]:
demand_trend = df.groupby("order_month", as_index=False).agg(
    units_sold=("quantity", "sum"),
    revenue_eur=("revenue", "sum"),
    orders=("order_id", "nunique"),
)
# PeriodIndex back to string for clean display and CSV export
demand_trend["order_month"] = demand_trend["order_month"].astype(str)
demand_trend.tail(12)

Monthly revenue line. Each Nov/Dec peak is about 3x the off-season baseline.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(demand_trend["order_month"], demand_trend["revenue_eur"]/1e6,
        color="#2E5266", linewidth=2, marker="o")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (€ million)")
ax.set_title("Monthly Revenue (2022-2024). Nov/Dec holiday spikes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. OTIF: Carrier On-Time Performance

OTIF (On-Time-In-Full) is the most important supply chain KPI. % of shipments that arrived by the promised date.

Aggregate shipments by carrier, get 3 things: shipment count, on-time %, avg delivery days. The percentage gap between best and worst carrier is the headline operational story.

In [ ]:
carrier_perf = shipments.groupby("carrier", as_index=False).agg(
    shipments=("shipment_id", "count"),
    on_time_pct=("on_time_delivery", lambda x: round(100 * x.mean(), 1)),
    avg_delivery_days=("actual_delivery_days", lambda x: round(x.mean(), 2)),
).sort_values("on_time_pct", ascending=False)
carrier_perf

OTIF bar chart with traffic-light colouring. Green >=70%, amber >=55%, red below. Dashed line marks the 70% benchmark most retailers use.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
# green >=70%, amber >=55%, red below. instant visual triage
colors = ["#2ECC71" if p >= 70 else "#F39C12" if p >= 55 else "#E74C3C"
          for p in carrier_perf["on_time_pct"]]
bars = ax.bar(carrier_perf["carrier"], carrier_perf["on_time_pct"], color=colors)
ax.axhline(70, color="green", linestyle="--", alpha=0.5, label="70% target")
ax.set_ylabel("On-time delivery %")
ax.set_title("OTIF by Carrier")
ax.set_ylim(0, 100)
for bar, val in zip(bars, carrier_perf["on_time_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, f"{val}%",
            ha="center", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Warehouse Fulfilment Speed

Each shipment fulfils from one of 6 warehouses. Joining shipments to orders to warehouses links every delivery back to its hub.

A slow warehouse is a bottleneck for every customer it serves. Isolating one outlier is directly actionable.

In [ ]:
wh_perf = shipments.merge(orders[["order_id", "warehouse_id"]], on="order_id")
wh_perf = wh_perf.merge(warehouses[["warehouse_id", "warehouse_name", "city"]], on="warehouse_id")

wh_summary = wh_perf.groupby(["warehouse_name", "city"], as_index=False).agg(
    shipments=("shipment_id", "count"),
    avg_delivery_days=("actual_delivery_days", lambda x: round(x.mean(), 2)),
    on_time_pct=("on_time_delivery", lambda x: round(100 * x.mean(), 1)),
).sort_values("avg_delivery_days")
wh_summary

Horizontal bars work better than vertical because warehouse names are too long for rotated labels.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(wh_summary["warehouse_name"], wh_summary["avg_delivery_days"],
               color="#3498DB")
ax.set_xlabel("Average delivery days")
ax.set_title("Fulfilment Speed by Warehouse")
for bar, val in zip(bars, wh_summary["avg_delivery_days"]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, f"{val}d",
            va="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Return Pattern Analysis

Returns are pure cost. Refunds + reverse logistics + restocking work. Knowing which categories return most and why drives directly actionable decisions.

The first table left-joins master DataFrame with returns. Every order kept, only returned ones have a non-null `return_reason`. Count of non-null / total = return rate per category.

Second analysis: aggregate by reason. The category with the highest return rate plus the dominant reason points at the single most valuable intervention. In e-commerce that's almost always "Wrong size" in Fashion, which justifies sizing tool investment.

In [ ]:
# left join. every order kept, returned ones have a return_reason
ret_by_cat = df.merge(returns[["order_id", "return_reason", "refund_amount"]],
                      on="order_id", how="left")
ret_by_cat["is_returned"] = ret_by_cat["return_reason"].notna()

cat_return_rate = ret_by_cat.groupby("category").agg(
    orders=("order_id", "nunique"),
    returns=("is_returned", "sum"),
).reset_index()
cat_return_rate["return_rate_pct"] = (100 * cat_return_rate["returns"] / cat_return_rate["orders"]).round(2)
cat_return_rate = cat_return_rate.sort_values("return_rate_pct", ascending=False)
cat_return_rate

Colour by severity. Red >15%, amber >7%, green below.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
colors = ["#E74C3C" if r > 15 else "#F39C12" if r > 7 else "#2ECC71"
          for r in cat_return_rate["return_rate_pct"]]
bars = ax.bar(cat_return_rate["category"], cat_return_rate["return_rate_pct"], color=colors)
ax.set_ylabel("Return rate %")
ax.set_title("Return Rate by Category")
ax.axhline(10, color="gray", linestyle="--", alpha=0.5, label="10% threshold")
for bar, val in zip(bars, cat_return_rate["return_rate_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f"{val}%",
            ha="center", fontweight="bold")
plt.xticks(rotation=30, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

Top return reasons. Knowing whether refunds are driven by sizing, damage, or quality determines which fix has the highest ROI.

In [ ]:
reason_summary = returns.groupby("return_reason", as_index=False).agg(
    return_count=("return_id", "count"),
    total_refund_eur=("refund_amount", "sum"),
)
reason_summary["pct_of_returns"] = (100 * reason_summary["return_count"]
                                    / reason_summary["return_count"].sum()).round(1)
reason_summary = reason_summary.sort_values("return_count", ascending=False)
reason_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(reason_summary["return_reason"], reason_summary["return_count"], color="#9B59B6")
ax.set_xlabel("Number of returns")
ax.set_title("Top Return Reasons")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 11. Forecast vs Actual: Accuracy

Forecast accuracy is a foundational demand planning metric. Standard formula:

**Accuracy = 1 - (sum of absolute errors / sum of actuals)**

I don't have a real forecasting model here, so the forecast is simulated by perturbing each actual monthly demand by +/- 15%. In production this would come from a model (Holt-Winters, Prophet, or a vendor feed).

The output ranks categories by predictability. Anything below 80% would typically get flagged for a model refresh.

In [ ]:
np.random.seed(42)

# actual monthly demand per category
actual = df.groupby(["category", "order_month"], as_index=False)["quantity"].sum()
actual.rename(columns={"quantity": "actual_qty"}, inplace=True)

# simulate forecast by perturbing actual by +/- 15%
# in production this would come from a real model
actual["forecast_qty"] = (actual["actual_qty"]
                          * np.random.uniform(0.85, 1.15, size=len(actual))).round().astype(int)

actual["error"]     = actual["actual_qty"] - actual["forecast_qty"]
actual["abs_error"] = actual["error"].abs()

# overall = 1 - (sum abs errors / sum actuals)
overall_accuracy = 1 - (actual["abs_error"].sum() / actual["actual_qty"].sum())
print(f"Overall forecast accuracy: {overall_accuracy*100:.2f}%")

# same per category to rank predictability
cat_accuracy = actual.groupby("category", as_index=False).apply(
    lambda g: pd.Series({
        "accuracy_pct": round(100 * (1 - g["abs_error"].sum() / g["actual_qty"].sum()), 2)
    }), include_groups=False
).sort_values("accuracy_pct", ascending=False)
cat_accuracy

## 12. Stockout Risk Proxy

No real inventory levels in this data, so direct stockout calc isn't possible. Using a velocity proxy instead. Products that sell most units per month consume buffer stock fastest, so they're the most exposed.

Velocity = `units_sold / months_active`. Top 20 highest-velocity SKUs are the ones inventory planners would watch most closely. In a real environment this would be combined with current stock on hand and supplier lead times to flag actual risk.

In [ ]:
# velocity = units sold / months the product has been active
product_velocity = df.groupby("product_id", as_index=False).agg(
    units_sold=("quantity", "sum"),
    revenue=("revenue", "sum"),
    category=("category", "first"),
)
product_velocity["months_active"] = df.groupby("product_id")["order_month"].nunique().values
product_velocity["units_per_month"] = (product_velocity["units_sold"]
                                       / product_velocity["months_active"]).round(1)

# top 20 highest-velocity SKUs. most exposed to stockouts
top_velocity = product_velocity.sort_values("units_per_month", ascending=False).head(20)
top_velocity[["product_id", "category", "units_sold", "units_per_month", "revenue"]]

## 13. Customer Geography: Revenue by City

Where the customer redistribution from Section 3 pays off.

One subtle pandas thing: both the master DataFrame and the customers table have a `city` column but they mean different things. `df.city` is the warehouse city (because warehouses were merged earlier). `customers.city` is the customer's city. Without `suffixes`, one silently overwrites the other.

In [ ]:
# merge customer city onto df. df already has 'city' (warehouse city), suffixes required
customer_geo = df.merge(
    customers[["customer_id", "city"]],
    on="customer_id",
    suffixes=("_warehouse", "_customer")
)

# group by customer city specifically. that's the geographic question being asked
city_revenue = customer_geo.groupby("city_customer", as_index=False).agg(
    customers=("customer_id", "nunique"),
    orders=("order_id", "nunique"),
    revenue_eur=("revenue", "sum"),
).sort_values("revenue_eur", ascending=False)

city_revenue = city_revenue.rename(columns={"city_customer": "city"})
city_revenue["pct_of_total"] = (100 * city_revenue["revenue_eur"]
                                / city_revenue["revenue_eur"].sum()).round(1)
city_revenue

Berlin, Hamburg, München should dominate, reflecting the population weights from Section 3.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(city_revenue["city"], city_revenue["revenue_eur"]/1e6, color="#1ABC9C")
ax.set_xlabel("Revenue (€ million)")
ax.set_title("Revenue by City")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 14. Export Files for Tableau

Write analysis tables to `tableau_exports/`. Small pre-aggregated files, much faster for Tableau to load than the raw 3.4M-row order_items table.

`master_data` is the only big one (wide source for filterable dashboards). The other 10 are summary tables, each feeding a specific chart.

In [ ]:
import os

EXPORT = "/content/drive/MyDrive/BlitzMart_Analytics/tableau_exports"
os.makedirs(EXPORT, exist_ok=True)

# master_data is the wide source for row-level dashboards. others are summaries
exports = {
    "master_data": df[["order_id", "order_date", "order_month", "customer_id",
                       "product_id", "product_name", "category", "warehouse_name",
                       "region", "city", "quantity", "unit_price", "revenue",
                       "gross_profit", "payment_method"]],
    "abc_products":       abc_products,
    "abc_customers":      customer_revenue,
    "demand_trend":       demand_trend,
    "carrier_otif":       carrier_perf,
    "warehouse_speed":    wh_summary,
    "return_by_category": cat_return_rate,
    "return_reasons":     reason_summary,
    "forecast_accuracy":  cat_accuracy,
    "product_velocity":   product_velocity,
    "city_revenue":       city_revenue,
}

for name, frame in exports.items():
    path = f"{EXPORT}/{name}.csv"
    frame.to_csv(path, index=False)
    print(f"  {name}.csv  ({len(frame):,} rows)")

print(f"\nAll exports saved to: {EXPORT}")

## 15. Key Findings Summary

Auto-generated summary from the exported tables. Generating from actual data (not hardcoded numbers) keeps the headlines in sync if the data ever changes.

Covers 9 areas: scale, product concentration, customer concentration, seasonality, carrier OTIF, warehouse, returns, forecast, geography.

In [ ]:
import pandas as pd

EXPORT = "/content/drive/MyDrive/BlitzMart_Analytics/tableau_exports"

# load all exports, each drives a section below
master       = pd.read_csv(f"{EXPORT}/master_data.csv")
abc_prods    = pd.read_csv(f"{EXPORT}/abc_products.csv")
abc_custs    = pd.read_csv(f"{EXPORT}/abc_customers.csv")
demand       = pd.read_csv(f"{EXPORT}/demand_trend.csv")
carrier      = pd.read_csv(f"{EXPORT}/carrier_otif.csv")
warehouse    = pd.read_csv(f"{EXPORT}/warehouse_speed.csv")
ret_cat      = pd.read_csv(f"{EXPORT}/return_by_category.csv")
ret_reason   = pd.read_csv(f"{EXPORT}/return_reasons.csv")
forecast     = pd.read_csv(f"{EXPORT}/forecast_accuracy.csv")
city         = pd.read_csv(f"{EXPORT}/city_revenue.csv")

print("="*72)
print("BLITZMART KEY FINDINGS")
print("="*72)

# scale
total_rev  = master["revenue"].sum()
total_ord  = master["order_id"].nunique()
total_cust = master["customer_id"].nunique()
aov = total_rev / total_ord
print(f"\nSCALE")
print(f"  EUR {total_rev/1e9:.2f}B total revenue across {total_ord:,} orders")
print(f"  {total_cust:,} active customers, AOV EUR {aov:,.0f}")

# product concentration
a_prods = (abc_prods["abc_class"] == "A").sum()
a_rev_pct = 100 * abc_prods[abc_prods["abc_class"] == "A"]["revenue"].sum() / abc_prods["revenue"].sum()
a_share_pct = 100 * a_prods / len(abc_prods)
print(f"\nPRODUCT CONCENTRATION (ABC)")
print(f"  {a_share_pct:.1f}% of products ({a_prods} A-class SKUs) drive {a_rev_pct:.1f}% of revenue")

# customer concentration
top_20_pct_count = int(len(abc_custs) * 0.2)
top_20_rev = abc_custs.nlargest(top_20_pct_count, "revenue")["revenue"].sum()
top_20_pct = 100 * top_20_rev / abc_custs["revenue"].sum()
print(f"\nCUSTOMER CONCENTRATION")
print(f"  Top 20% of customers ({top_20_pct_count:,}) generate {top_20_pct:.1f}% of revenue")

# seasonality
demand["order_month"] = pd.to_datetime(demand["order_month"])
demand["month_num"] = demand["order_month"].dt.month
peak_months = demand.groupby("month_num")["revenue_eur"].mean()
normal_avg  = peak_months[peak_months.index.isin([1,2,3,4,5,8,9,10])].mean()
peak_avg    = peak_months[peak_months.index.isin([11,12])].mean()
print(f"\nSEASONALITY")
print(f"  Nov/Dec average revenue is {peak_avg/normal_avg:.1f}x higher than off-season")
print(f"  Peak monthly revenue: EUR {peak_avg/1e6:.1f}M vs normal EUR {normal_avg/1e6:.1f}M")

# carrier
best_c, worst_c = carrier.iloc[0], carrier.iloc[-1]
print(f"\nCARRIER PERFORMANCE (OTIF)")
print(f"  {best_c['carrier']} leads at {best_c['on_time_pct']}% on-time")
print(f"  {worst_c['carrier']} lags at {worst_c['on_time_pct']}% on-time")
print(f"  Gap: {best_c['on_time_pct'] - worst_c['on_time_pct']:.1f} percentage points")

# warehouse
best_wh, worst_wh = warehouse.iloc[0], warehouse.iloc[-1]
print(f"\nWAREHOUSE PERFORMANCE")
print(f"  Best:  {best_wh['warehouse_name']} at {best_wh['on_time_pct']}% OTIF, {best_wh['avg_delivery_days']}d avg")
print(f"  Worst: {worst_wh['warehouse_name']} at {worst_wh['on_time_pct']}% OTIF, {worst_wh['avg_delivery_days']}d avg")

# returns
worst_ret, best_ret = ret_cat.iloc[0], ret_cat.iloc[-1]
top_reason = ret_reason.iloc[0]
print(f"\nRETURN PATTERNS")
print(f"  Highest return rate: {worst_ret['category']} at {worst_ret['return_rate_pct']}%")
print(f"  Lowest return rate:  {best_ret['category']} at {best_ret['return_rate_pct']}%")
print(f"  #1 return reason: '{top_reason['return_reason']}' ({top_reason['pct_of_returns']}% of returns)")

# forecast accuracy
best_fcst, worst_fcst = forecast.iloc[0], forecast.iloc[-1]
print(f"\nFORECAST ACCURACY")
print(f"  Most predictable:  {best_fcst['category']} at {best_fcst['accuracy_pct']}%")
print(f"  Least predictable: {worst_fcst['category']} at {worst_fcst['accuracy_pct']}%")

# geography
top3_pct = city.head(3)["pct_of_total"].sum()
top3_cities = ", ".join(city.head(3)["city"].tolist())
print(f"\nGEOGRAPHIC CONCENTRATION")
print(f"  Top 3 cities ({top3_cities}) drive {top3_pct:.1f}% of revenue")
print(f"  Top city: {city.iloc[0]['city']} at {city.iloc[0]['pct_of_total']}%")

print("\n" + "="*72)

---

## Wrap-up

12 industry-standard analytical techniques applied to the cleaned BlitzMart dataset. Produced the chart outputs visible in the notebook plus the 11 aggregated CSVs that feed the Tableau dashboards.

Findings printed at the end are the headline numbers referenced in the project's Business Insights document and across the live dashboards. Together with the SQL analysis in Notebook 2, this completes the end-to-end analytical workflow from raw data to business recommendations.